In [1]:
# Imports, paths and load staging data

from pathlib import Path
import re
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

RAW_FILE = (
    PROJECT_ROOT
    / "Raw_Data"
    / "NFHS_5_India_Districts_Factsheet_Data.xls"
)
STAGING_FILE = PROJECT_ROOT / "Staging" / "nfhs_staging.csv"

CLEAN_DIR = PROJECT_ROOT / "Cleaned_Data"
REPORT_DIR = (
    PROJECT_ROOT
    / "Reports"
    / "03_data_cleaning_and_validation"
)

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_FILE = CLEAN_DIR / "nfhs_cleaned.csv"
CLEANING_LOG_FILE = CLEAN_DIR / "cleaning_log.csv"
QUALITY_REPORT_FILE = REPORT_DIR / "data_quality_issues.csv"

if not STAGING_FILE.exists():
    raise FileNotFoundError(
        f"Staging file not found: {STAGING_FILE}. Run Notebook 2 first."
    )

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw source file not found: {RAW_FILE}")

print("Notebook 3 paths configured.")
print(f"Input: {STAGING_FILE}")
print(f"Cleaned output: {CLEAN_FILE}")
print(f"Reports: {REPORT_DIR}")

Notebook 3 paths configured.
Input: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Staging\nfhs_staging.csv
Cleaned output: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Cleaned_Data\nfhs_cleaned.csv
Reports: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\03_data_cleaning_and_validation


In [2]:
# Load data and identify percentage indicators

df_staging = pd.read_csv(
    STAGING_FILE,
    low_memory=False
)

df_clean = df_staging.copy()

df_raw_headers = pd.read_excel(
    RAW_FILE,
    sheet_name="Sheet1",
    engine="xlrd",
    nrows=0
)

def standardize_column_name(column):
    """Standardise column names to lowercase snake_case."""
    column = re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        str(column).strip().lower()
    )
    return column.strip("_")


# Reproduce Notebook 2's column naming so source headers
# can be matched to their staged column names.
source_column_names = []
seen = {}

for original_column in df_raw_headers.columns:
    name = standardize_column_name(original_column)
    count = seen.get(name, 0)

    source_column_names.append(
        name if count == 0 else f"{name}_{count}"
    )
    seen[name] = count + 1

percentage_columns = []

for original_column, staged_column in zip(
    df_raw_headers.columns,
    source_column_names
):
    original_text = str(original_column).lower()

    if (
        "%" in original_text
        or "percent" in original_text
        or "percentage" in original_text
    ):
        if staged_column in df_clean.columns:
            percentage_columns.append(staged_column)

required_columns = {"district_names", "state_ut"}

if not required_columns.issubset(df_clean.columns):
    raise ValueError("Required district/state columns are missing.")

if df_clean.shape != df_staging.shape:
    raise ValueError("Staging structure changed unexpectedly.")

print(f"Rows loaded: {len(df_clean):,}")
print(f"Columns loaded: {len(df_clean.columns):,}")
print(f"Percentage indicators identified: {len(percentage_columns):,}")
print(percentage_columns)

Rows loaded: 707
Columns loaded: 109
Percentage indicators identified: 101
['female_population_age_6_years_and_above_who_ever_attended_school', 'population_below_age_15_years', 'children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority', 'deaths_in_the_last_3_years_registered_with_the_civil_authority', 'population_living_in_households_with_electricity', 'population_living_in_households_with_an_improved_drinking_water_source1', 'population_living_in_households_that_use_an_improved_sanitation_facility2', 'households_using_clean_fuel_for_cooking3', 'households_using_iodized_salt', 'households_with_any_usual_member_covered_under_a_health_insurance_financing_scheme', 'children_age_5_years_who_attended_pre_primary_school_during_the_school_year_2019_20', 'women_age_15_49_who_are_literate4', 'women_age_15_49_with_10_or_more_years_of_schooling', 'women_age_20_24_years_married_before_age_18_years', 'births_in_the_5_years_preceding_the_survey_that_are_third_or_higher_order', 

In [3]:
# Standardise missing markers and data types

cleaning_log = []

unavailable_markers = [
    "*", "-", "NA", "N/A", "NULL", "NONE"
]

identifier_columns = ["district_names", "state_ut"]

for column in identifier_columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Convert known unavailable markers to missing values.
for column in df_clean.columns:
    if column in identifier_columns:
        continue

    text_values = df_clean[column].astype("string").str.strip()
    normalized_values = text_values.str.upper()

    unavailable_mask = (
        normalized_values.isin(unavailable_markers)
        | text_values.eq("")
    ).fillna(False)

    unavailable_count = int(unavailable_mask.sum())

    if unavailable_count:
        df_clean.loc[unavailable_mask, column] = np.nan

        cleaning_log.append({
            "column": column,
            "action": "Replace unavailable markers with missing",
            "affected_rows": unavailable_count,
            "reason": "Recognised unavailable-value marker"
        })

    # Convert indicator values to numeric where possible.
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

print("Unavailable markers standardised.")
print(f"Numeric indicator columns processed: {len(df_clean.columns) - 2:,}")
print(f"Missing values after conversion: {df_clean.isna().sum().sum():,}")

Unavailable markers standardised.
Numeric indicator columns processed: 107
Missing values after conversion: 4,132


In [4]:
# Identify and handle invalid indicator values

quality_issues = []

# Validate percentage indicators.
for column in percentage_columns:
    numeric_values = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

    invalid_mask = (
        numeric_values.notna()
        & ~numeric_values.between(0, 100)
    )

    for row_index in df_clean.index[invalid_mask]:
        quality_issues.append({
            "row_index": int(row_index),
            "district_names": df_clean.at[row_index, "district_names"],
            "state_ut": df_clean.at[row_index, "state_ut"],
            "column": column,
            "observed_value": numeric_values.at[row_index],
            "issue_type": "Percentage outside 0-100",
            "action": "Replaced with missing; verify source"
        })

    affected_count = int(invalid_mask.sum())

    if affected_count:
        df_clean.loc[invalid_mask, column] = np.nan

        cleaning_log.append({
            "column": column,
            "action": "Set out-of-range percentage values to missing",
            "affected_rows": affected_count,
            "reason": "Percentage indicators must be within 0-100"
        })

# Flag negative values only in indicators whose names indicate
# non-negative expenditure or sex-ratio measures.
non_negative_keywords = [
    "expenditure",
    "sex_ratio"
]

for column in df_clean.columns:
    if column in identifier_columns:
        continue

    if not any(
        keyword in column.lower()
        for keyword in non_negative_keywords
    ):
        continue

    numeric_values = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

    invalid_mask = (
        numeric_values.notna()
        & numeric_values.lt(0)
    )

    for row_index in df_clean.index[invalid_mask]:
        quality_issues.append({
            "row_index": int(row_index),
            "district_names": df_clean.at[row_index, "district_names"],
            "state_ut": df_clean.at[row_index, "state_ut"],
            "column": column,
            "observed_value": numeric_values.at[row_index],
            "issue_type": "Negative value in non-negative indicator",
            "action": "Replaced with missing; verify source"
        })

    affected_count = int(invalid_mask.sum())

    if affected_count:
        df_clean.loc[invalid_mask, column] = np.nan

        cleaning_log.append({
            "column": column,
            "action": "Set negative indicator values to missing",
            "affected_rows": affected_count,
            "reason": "Indicator is expected to be non-negative"
        })

quality_report = pd.DataFrame(
    quality_issues,
    columns=[
        "row_index",
        "district_names",
        "state_ut",
        "column",
        "observed_value",
        "issue_type",
        "action"
    ]
)

quality_report.to_csv(
    QUALITY_REPORT_FILE,
    index=False
)

print(f"Quality issues identified: {len(quality_report):,}")
print(f"Quality report: {QUALITY_REPORT_FILE}")

if not quality_report.empty:
    display(
        quality_report.groupby("issue_type")
        .size()
        .reset_index(name="issue_count")
    )
else:
    print("No values violated the configured indicator rules.")

Quality issues identified: 5,069
Quality report: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\03_data_cleaning_and_validation\data_quality_issues.csv


,issue_type,issue_count
0,Negative value in non-negative indicator,16
1,Percentage outside 0-100,5053


In [5]:
# Correct the state spelling and record the action

source_state = "Maharastra"
corrected_state = "Maharashtra"

correction_mask = df_clean["state_ut"].eq(source_state).fillna(False)
correction_count = int(correction_mask.sum())

if correction_count:
    df_clean.loc[correction_mask, "state_ut"] = corrected_state

    cleaning_log.append({
        "column": "state_ut",
        "action": "Standardise state spelling",
        "affected_rows": correction_count,
        "reason": f"{source_state} changed to {corrected_state}"
    })

print(f"State spelling corrections: {correction_count}")
print(
    "Remaining Maharastra rows:",
    int(df_clean["state_ut"].eq(source_state).sum())
)

State spelling corrections: 36
Remaining Maharastra rows: 0


In [6]:
# Validate identifiers and district/state combinations

# Identify missing or blank identifiers.
missing_identifier_mask = (
    df_clean["district_names"].isna()
    | df_clean["district_names"].eq("")
    | df_clean["state_ut"].isna()
    | df_clean["state_ut"].eq("")
)

missing_identifiers = df_clean.loc[
    missing_identifier_mask,
    ["district_names", "state_ut"]
].copy()

# Identify duplicate district/state combinations.
duplicate_pairs = df_clean.loc[
    df_clean.duplicated(
        subset=["district_names", "state_ut"],
        keep=False
    ),
    ["district_names", "state_ut"]
].sort_values(["state_ut", "district_names"])

# Identify district names associated with multiple states.
district_state_conflicts = (
    df_clean.dropna(subset=["district_names", "state_ut"])
    .groupby("district_names")["state_ut"]
    .nunique()
    .loc[lambda counts: counts > 1]
    .rename("state_count")
    .reset_index()
)

missing_identifiers.to_csv(
    REPORT_DIR / "missing_identifiers.csv",
    index=False
)

duplicate_pairs.to_csv(
    REPORT_DIR / "duplicate_district_state_pairs.csv",
    index=False
)

district_state_conflicts.to_csv(
    REPORT_DIR / "district_names_multiple_states.csv",
    index=False
)

print(f"Rows with missing identifiers: {len(missing_identifiers):,}")
print(f"Rows in duplicate pairs: {len(duplicate_pairs):,}")
print(
    "District names occurring in multiple states:",
    len(district_state_conflicts)
)

display(district_state_conflicts)

Rows with missing identifiers: 0
Rows in duplicate pairs: 0
District names occurring in multiple states: 8


,district_names,state_count
0,Aurangabad,2
1,Balrampur,2
2,Bijapur,2
3,Bilaspur,2
4,Chandel,2
5,Hamirpur,2
6,Pratapgarh,2
7,Raigarh,2


In [7]:
# Save the cleaned data and cleaning log

# Confirm that cleaning has not changed the dataset structure.
assert df_clean.shape == df_staging.shape, (
    "Cleaning changed the row or column count."
)

if not missing_identifiers.empty:
    raise ValueError(
        "Missing district/state identifiers remain. "
        "Review the missing_identifiers report."
    )

if not duplicate_pairs.empty:
    raise ValueError(
        "Duplicate district-state pairs remain. "
        "Review the duplicate_district_state_pairs report."
    )

if df_clean["state_ut"].eq("Maharastra").any():
    raise ValueError("Uncorrected Maharastra values remain.")

df_clean.to_csv(
    CLEAN_FILE,
    index=False
)

pd.DataFrame(
    cleaning_log,
    columns=["column", "action", "affected_rows", "reason"]
).to_csv(
    CLEANING_LOG_FILE,
    index=False
)

print(f"Cleaned dataset saved: {CLEAN_FILE}")
print(f"Cleaning log saved: {CLEANING_LOG_FILE}")
print(f"Quality report saved: {QUALITY_REPORT_FILE}")
print(f"Cleaned shape: {df_clean.shape}")

Cleaned dataset saved: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Cleaned_Data\nfhs_cleaned.csv
Cleaning log saved: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Cleaned_Data\cleaning_log.csv
Quality report saved: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\03_data_cleaning_and_validation\data_quality_issues.csv
Cleaned shape: (707, 109)


In [8]:
# Reload and verify saved outputs

df_verified = pd.read_csv(
    CLEAN_FILE,
    low_memory=False
)

assert df_verified.shape == df_staging.shape
assert df_verified["district_names"].notna().all()
assert df_verified["state_ut"].notna().all()

assert not df_verified.duplicated(
    ["district_names", "state_ut"]
).any()

assert not df_verified["state_ut"].eq("Maharastra").any()

print("Final saved-file validation passed.")
print(f"Rows: {len(df_verified):,}")
print(f"Columns: {len(df_verified.columns):,}")
print(
    "Maharashtra rows:",
    int(df_verified["state_ut"].eq("Maharashtra").sum())
)
print(f"Reports folder: {REPORT_DIR}")

Final saved-file validation passed.
Rows: 707
Columns: 109
Maharashtra rows: 36
Reports folder: C:\PORTFOLIO\NFHS-5-Healthcare-Analytics-District-Intelligence\Reports\03_data_cleaning_and_validation
